In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import math
import os
import ws3.opt
import pickle
import numpy as np
from math import pi

In [2]:
from util import generate_radar_chart

In [3]:
obj_mode = ["Max_hv", "Max_st", "Min_em", "Min_ha"]

In [4]:
data_max_hv= {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    # "Harvest Volume": [9.5385144e+07, 1.3344616e+08, 8.584663087e+07, 7.6308117e+07, 6.676960411e+07, 5.723109073e+07, 4.7692577e+07, 3.8154064e+07, 2.8615551e+07, 1.9077037e+07, 9.5385238e+06],
    "Net Emission": [-10187414, 2860992, -20190655, -37717399, -53571451, -69020879, -83747319, -96901488, -116527826, -127261518, -141630206],
    "Social Indicator": [100842, 141080, 90757, 80673, 70589, 60505, 50421, 40336, 30252, 20168, 10084],
    "Economic Indicator": [3438933, 4811151, 3095039, 2751146, 2407253, 2063360, 1719466, 1375573, 1031680, 687786, 343893],
    "Species Indicator": [0.7329, 0.7509, 0.7607, 0.7514, 0.7445, 0.7401, 0.7375, 0.7275, 0.7328, 0.7365, 0.7399],
    "Old Growth Indicator": [196384, 166327, 222922.6, 247449, 286878.4, 324882.5, 367419, 420221.3, 462754.7, 502643, 535463.1]
}

data_min_ha = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    # "Harvest Area": [3.8189149e+05, None, 3.3325811e+05, 2.8743943e+05, 2.4404208e+05, 2.0333523e+05, 1.6452611e+05, 1.2681471e+05, 9.060671199e+04, 5.7354080e+04, 2.7180641e+04],
    "Net Emissions": [-27541849, -155624647, -42297895, -55475110, -69237782, -83282990, -96462861, -109037924, -121829519, -133928525, -144780424],
    "Social Indicator": [92157, None, 82941, 73725, 64510, 55294, 46078, 36862, 27647, 18431, 9215],
    "Economic Indicator": [3142756, None, 2828481, 2514205, 2199929, 1885654, 1571378, 1257102, 942827, 628551, 314275],
    "Species Indicator": [0.7543, 0.7457, 0.7534, 0.7491, 0.7466, 0.7443, 0.7456, 0.7431, 0.7487, 0.7489, 0.7502],
    "Old Growth Indicator": [303678.5, 580504.3, 334075.7, 364650.4, 393684.8, 423066.2, 451206.5, 479354.5, 505137.4, 530289.9, 556633.9]
}


data_max_st = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    # "Carbon Stock": [1.993899100e+08, 2.213324476e+08, 2.023201210e+08, 2.049510923e+08, 2.073555872e+08,
    #                  2.0959373e+08, 2.1161157e+08, 2.1346753e+08, 2.1511490e+08, 2.1654530e+08, 2.1772600e+08],
    "Net Emissions": [-34255783, -117907115, -49672695, -64270430, -78526028, -91980728, -104984288, -117144321,
                      -128555316, -139248380, -148466128],
    "Social Indicator": [92157, 42718, 83272, 74497, 65185, 55858, 46351, 37113, 27854, 18498, 9293],
    "Economic Indicator": [3142756, 1456798, 2839766, 2540532, 2222965, 1904896, 1580700, 1265647, 949882,
                                630844, 316931],
    "Species Indicator": [0.7534, 0.7618, 0.7507, 0.7484, 0.7441, 0.7440, 0.7438, 0.7428, 0.7419, 0.7425,
                                     0.7429],
    "Old Growth Indicator": [317668.7, 556478.12, 343022.16, 366396.3, 390767.51, 417504.07, 445561.57,
                                            473603.09, 500597.34, 526910.75, 552527.14]
}

data_min_em = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    # "Net Emission": [-8.4556995e+07, -1.6051199e+08, -9.6237592e+07, -1.0688514e+08, -1.1650972e+08,
    #                  -1.2533166e+08, -1.3318654e+08, -1.4030201e+08, -1.466282322e+08, -1.5177528e+08,
    #                  -1.549826706e+08],
    "Net Emissions": [-33180983, -151871771, -48819211, -63980807, -78384629, -92468485, -105856916,
                      -118833104, -130275827, -140664223, -148892042],
    "Social Indicator": [92157, 13888, 82941, 73725, 64510, 55294, 46078, 36862, 27647, 18443, 9408],
    "Economic Indicator": [3142756, 473621, 2828481, 2514205, 2199929, 1885654, 1571378, 1257102, 942827,
                                628976, 320857],
    "Species Indicator": [0.7258, 0.7421, 0.7137, 0.7063, 0.6987, 0.6929, 0.6930, 0.6971, 0.7071, 0.7228,
                                     0.7373],
    "Old Growth Indicator": [214539.2, 516820.07, 242967.94, 275445.01, 311282.5, 349133.9, 384958.37,
                                            422547.95, 459147.55, 498725.47, 538769.88]
}

In [6]:
generate_radar_chart(data_max_hv, output_dir="./plots/fig", case_study="equitysilver", obj_mode="Max_hv")

Chart saved at: ./plots/fig/equitysilver/equitysilver_Max_hv_radar_chart.svg


In [7]:
generate_radar_chart(data_min_ha, output_dir="./plots/fig", case_study="equitysilver", obj_mode="Min_ha")

Chart saved at: ./plots/fig/equitysilver/equitysilver_Min_ha_radar_chart.svg


In [7]:
generate_radar_chart(data_max_st, output_dir="./plots/fig", case_study="equitysilver", obj_mode="Max_st")

Chart saved at: ./plots/fig/equitysilver/equitysilver_Min_ha_radar_chart.svg


In [8]:
generate_radar_chart(data_min_em, output_dir="./plots/fig", case_study="equitysilver", obj_mode="Min_em")

Chart saved at: ./plots/fig/equitysilver/equitysilver_Min_em_radar_chart.svg
